# A toy example for HW7 Bert QA

Source code from:  
- [ML2021 hw7 English version](https://www.youtube.com/watch?v=SnqP5HnOBD8)
- [李宏毅ML2021 HW7 BERT-Question Answering_hw7 李宏毅 report questions-CSDN博客](https://blog.csdn.net/qq_39610915/article/details/119913009)

## Install Transfomer

In [1]:
#!pip install transformers==4.5.0
!pip install transformers

  Using cached transformers-4.5.0-py3-none-any.whl.metadata (41 kB)
  Using cached tokenizers-0.10.3.tar.gz (212 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Using cached transformers-4.5.0-py3-none-any.whl (2.1 MB)
Failed to build tokenizers


  error: subprocess-exited-with-error
  
  exit code: 1
  
  [49 lines of output]
  running bdist_wheel
  running build
  running build_py
  creating build\lib.win-amd64-cpython-310\tokenizers
  copying py_src\tokenizers\__init__.py -> build\lib.win-amd64-cpython-310\tokenizers
  creating build\lib.win-amd64-cpython-310\tokenizers\models
  copying py_src\tokenizers\models\__init__.py -> build\lib.win-amd64-cpython-310\tokenizers\models
  creating build\lib.win-amd64-cpython-310\tokenizers\decoders
  copying py_src\tokenizers\decoders\__init__.py -> build\lib.win-amd64-cpython-310\tokenizers\decoders
  creating build\lib.win-amd64-cpython-310\tokenizers\normalizers
  copying py_src\tokenizers\normalizers\__init__.py -> build\lib.win-amd64-cpython-310\tokenizers\normalizers
  creating build\lib.win-amd64-cpython-310\tokenizers\pre_tokenizers
  copying py_src\tokenizers\pre_tokenizers\__init__.py -> build\lib.win-amd64-cpython-310\tokenizers\pre_tokenizers
  creating build\lib.win-amd64-c

## Import Package

In [2]:
import torch
from transformers import AdamW, BertTokenizerFast, BertForQuestionAnswering

## Load Model and Tokenizer

In [3]:
# model_name can be either: models in huggingface model hub or models saved using save_pretrained
model_name = 'bert-base-chinese'
model = BertForQuestionAnswering.from_pretrained(model_name)

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
chi_tokenizer = BertTokenizerFast.from_pretrained('bert-base-chinese')
eng_tokenizer = BertTokenizerFast.from_pretrained('bert-base-cased')

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

D:\Users\ycho\anaconda3\envs\common\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ycho\.cache\huggingface\hub\models--bert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

## Tokenize

In [5]:
chi_paragraph = '李弘毅幾班大金。2021 ML'
tokens = chi_tokenizer.tokenize(chi_paragraph)
print(tokens)
chi_tokenizer.convert_tokens_to_ids(tokens)

['李', '弘', '毅', '幾', '班', '大', '金', '。', '2021', '[UNK]']


[3330, 2473, 3675, 2407, 4408, 1920, 7032, 511, 9960, 100]

In [6]:
eng_paragraph = 'Lee Hung-yi which class Daikin.'
tokens = eng_tokenizer.tokenize(eng_paragraph)
print(tokens)
eng_tokenizer.convert_tokens_to_ids(tokens)

['Lee', 'Hung', '-', 'y', '##i', 'which', 'class', 'Dai', '##kin', '.']


[2499, 26157, 118, 194, 1182, 1134, 1705, 23084, 4314, 119]

## Encode vs Decode

In [7]:
question = '李弘毅幾班?'
paragraph = '李弘毅幾班大金'
encoded = chi_tokenizer.encode(question, paragraph)
decoded = chi_tokenizer.decode(encoded)
print(encoded)
print(decoded)

[101, 3330, 2473, 3675, 2407, 4408, 136, 102, 3330, 2473, 3675, 2407, 4408, 1920, 7032, 102]
[CLS] 李 弘 毅 幾 班? [SEP] 李 弘 毅 幾 班 大 金 [SEP]


## Model Inputs

In [8]:
inputs = chi_tokenizer(question, paragraph, return_tensors='pt')
# Indices of input sequence tokens in the vocabulary
print('Input ids:      ', inputs['input_ids'])
# Segment token indices to indicate first and second portions of the inputs. Indices are selected in [0, 1]
print('Token type ids: ', inputs['token_type_ids'])
# Mask to avoid performing attention on padding token indices. Mask values selected in [0, 1]
print('Attention mask: ', inputs['attention_mask'])

Input ids:       tensor([[ 101, 3330, 2473, 3675, 2407, 4408,  136,  102, 3330, 2473, 3675, 2407,
         4408, 1920, 7032,  102]])
Token type ids:  tensor([[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1]])
Attention mask:  tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


## Testing (Chines)

In [11]:
question = '李弘毅幾班?'
paragraph = '李弘毅幾班大金。'
inputs = chi_tokenizer(question, paragraph, return_tensors='pt')

with torch.no_grad():
    output = model(**inputs)
# output = model(input_ids=inputs['input_ids'], token_type_ids=inputs['token_type_ids'], attention_mask=inputs['attention_mask'])

print("start_logits: ")
print(output.start_logits)

print("end_logits: ")
print(output.end_logits)

start = torch.argmax(output.start_logits) # 返回dim维度上张量最大值的索引。
end = torch.argmax(output.end_logits)
print("start position: ", start.item()) # 一个元素张量可以用x.item()得到元素值
print("end position:   ", end.item())

# 获取预测的start和end的token的id
predict_id = inputs['input_ids'][0][start : end + 1]
print("predict_id:     ", predict_id)
# 根据id解码出原文
predict_answer = chi_tokenizer.decode(predict_id)
print("predict_answer: ", predict_answer)

start_logits: 
tensor([[-0.6731, -0.3050, -1.1232, -1.0862, -0.6898, -0.5899, -0.4108, -1.3474,
         -0.7844, -1.2042, -1.1501, -0.3807, -0.7527,  1.2468,  0.2576,  0.3084,
         -1.3474]])
end_logits: 
tensor([[-0.8373, -1.4521, -1.9488, -2.0090, -1.7208, -0.8393, -0.8054, -1.7115,
         -1.9284, -1.8353, -2.0257, -1.5646, -1.1697, -0.7927,  0.6471, -0.6123,
         -1.7115]])
start position:  13
end position:    14
predict_id:      tensor([1920, 7032])
predict_answer:  大 金


## Training (Chinese)
For Question Answering, loss is the sum of cross entropy between model prediction and correct answer

In [10]:
# 指定正确答案的开始结束位置，13
output = model(**inputs, start_positions=torch.tensor([13]), end_positions=torch.tensor([14]))
print("loss: ", output.loss)

optimizer = AdamW(model.parameters(), lr=1e-4)
output.loss.backward()
optimizer.step()

loss:  tensor(2.8933, grad_fn=<DivBackward0>)


D:\Users\ycho\anaconda3\envs\common\lib\site-packages\transformers\optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


## Testing (English)

In [29]:
question = 'Why does Jeanie like Tom?'
paragraph = 'Jeanie likes Tom because Tom is good at deep learning.'
inputs = eng_tokenizer(question, paragraph, return_tensors='pt')

with torch.no_grad():
    output = model(**inputs)
# output = model(input_ids=inputs['input_ids'], token_type_ids=inputs['token_type_ids'], attention_mask=inputs['attention_mask'])

print("start_logits: ")
print(output.start_logits)

print("end_logits: ")
print(output.end_logits)

start = torch.argmax(output.start_logits) # 返回dim维度上张量最大值的索引。
end = torch.argmax(output.end_logits)
print("start position: ", start.item()) # 一个元素张量可以用x.item()得到元素值
print("end position:   ", end.item())

# 获取预测的start和end的token的id
predict_id = inputs['input_ids'][0][start : end + 1]
print("predict_id:     ", predict_id)
# 根据id解码出原文
predict_answer = eng_tokenizer.decode(predict_id)
print("predict_answer: ", predict_answer)

start_logits: 
tensor([[-1.0182, -0.3656, -0.6646, -0.7164, -0.5520, -0.4693,  0.1982, -0.7487,
         -1.8145, -0.3580, -0.0823,  0.0291,  0.9856,  0.8027,  1.2006,  0.3090,
          0.3870,  0.2441,  0.4446,  0.3205, -0.3033, -1.8145]])
end_logits: 
tensor([[-2.1099, -2.5696, -2.5130, -2.6114, -2.4110, -2.4309, -1.5790, -2.4758,
         -3.3955, -2.7388, -2.4877, -1.8952, -0.3028, -0.5725, -0.3787, -1.4711,
         -1.4353, -0.8224,  1.0962,  3.2624, -1.8353, -3.3955]])
start position:  14
end position:    19
predict_id:      tensor([2545, 1110, 1363, 1120, 1996, 3776])
predict_answer:  Tom is good at deep learning


## Training (English)
For Question Answering, loss is the sum of cross entropy between model prediction and correct answer

In [23]:
inputs = eng_tokenizer(question, paragraph, return_tensors='pt')
# Indices of input sequence tokens in the vocabulary
print('Input ids:      ', inputs['input_ids'])
# Segment token indices to indicate first and second portions of the inputs. Indices are selected in [0, 1]
print('Token type ids: ', inputs['token_type_ids'])
# Mask to avoid performing attention on padding token indices. Mask values selected in [0, 1]
print('Attention mask: ', inputs['attention_mask'])

Input ids:       tensor([[ 101, 2009, 1674, 2893, 1663, 1176, 2545,  136,  102, 2893, 1663, 7407,
         2545, 1272, 2545, 1110, 1363, 1120, 1996, 3776,  119,  102]])
Token type ids:  tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
Attention mask:  tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [27]:
# 获取预测的start和end的token的id
answer_id = inputs['input_ids'][0][14 : 19 + 1]
print("answer_id:     ", answer_id)
# 根据id解码出原文
answer_answer = eng_tokenizer.decode(answer_id)
print("answer_answer: ", answer_answer)

answer_id:      tensor([2545, 1110, 1363, 1120, 1996, 3776])
answer_answer:  Tom is good at deep learning


In [28]:
# 指定正确答案的开始结束位置，14
output = model(**inputs, start_positions=torch.tensor([14]), end_positions=torch.tensor([19]))
print("loss: ", output.loss)

optimizer = AdamW(model.parameters(), lr=1e-4)
output.loss.backward()
optimizer.step()

loss:  tensor(2.0983, grad_fn=<DivBackward0>)


## Other Test Topic

(See https://huggingface.co/docs/accelerate/usage_guides/gradient_accumulation)

In [30]:
# define toy inputs and labels
x = torch.tensor([1., 2., 3., 4., 5., 6., 7., 8.])
y = torch.tensor([2., 4., 6., 8., 10., 12., 14., 16.])
gradient_accumulation_steps = 4
batch_size = len(x) // gradient_accumulation_steps

print ("len(x):", len(x))
print ("batch_size:", batch_size)

len(x): 8
batch_size: 2
